> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataVisualization/blob/main/notebooks/02a_Layers_Labels_and_Basemaps.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Layers, Labels and Basemaps

**Series:** Cartography in Python (notebook 1 of 2)
**Time:** about 30 minutes

**In this notebook you will:**
- Stack several layers in a deliberate order (z-order, transparency, outlines)
- Zoom a map to the area that matters
- Label features legibly with halos, and choose which to label
- Add a basemap with contextily, and build a legend that covers every layer

**You should already know:** choropleth maps, classification and palettes (notebook 01c of this course) and the Map Design module.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## From a chart to a map people can read

Notebook 01c produced correct choropleths. A finished map also needs **context**: where is this, what are the
landmarks, what is on top of what. This notebook builds one map of Lafayette and West Lafayette, Indiana, layer by
layer: households without a car by census tract, with parks and city limits on top, a few labels and a basemap.

Visual hierarchy (Map Design module) decides every choice below: the **subject** (the choropleth) is strong, the
**context** (basemap, outlines) is quiet, and the **reference** (labels) is readable but small.

In [ ]:
# mapclassify contextily comes installed in the SpatialTurn GIS kernel, so there is nothing to install.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts_acs.geojson").to_crs(epsg=26916)
parks = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson").to_crs(epsg=26916)
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson").to_crs(epsg=26916)
tracts["pct_no_vehicle"] = np.where(tracts["households"] > 0,
                                    tracts["hh_no_vehicle"] / tracts["households"] * 100, np.nan)

print(len(tracts), "tracts,", len(parks), "parks")

## Step 1: Layer order

Each `plot()` call draws on the same axes. Later layers are drawn on top, and `zorder` makes the order explicit.
Polygons on top of a choropleth need either transparency (`alpha`) or no fill at all.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
tracts.plot(ax=ax, column="pct_no_vehicle", scheme="NaturalBreaks", k=5, cmap="YlOrRd",
            edgecolor="white", linewidth=0.4, zorder=1,
            missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white"})
parks.plot(ax=ax, color="#31a354", edgecolor="none", alpha=0.8, zorder=2)
city.boundary.plot(ax=ax, color="#252525", linewidth=1.2, linestyle="--", zorder=3)
ax.set_axis_off()
plt.show()

## Step 2: Zoom to what matters

The county's large rural tracts dominate the frame, and the city, where the story is, is small. Set the axes
limits to the city's extent plus a margin.

In [ ]:
xmin, ymin, xmax, ymax = city.total_bounds
pad = 1500                                   # meters
EXTENT = (xmin - pad, xmax + pad, ymin - pad, ymax + pad)

def zoom(ax):
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])

## Step 3: Labels with halos

Labels compete with everything underneath them. A thin **halo** (an outline in the background color) keeps text
readable over any color, and is the most useful single trick in cartographic labeling. Label only what helps
the reader: here, the largest parks inside the view.

In [ ]:
import matplotlib.patheffects as pe
HALO = [pe.withStroke(linewidth=3, foreground="white")]

view = parks.cx[EXTENT[0]:EXTENT[1], EXTENT[2]:EXTENT[3]].copy()
view["ha"] = view.area / 1e4
big = view.sort_values("ha", ascending=False).head(6)
print(big[["name", "ha"]].round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
tracts.plot(ax=ax, column="pct_no_vehicle", scheme="NaturalBreaks", k=5, cmap="YlOrRd",
            edgecolor="white", linewidth=0.4, zorder=1,
            missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white"})
parks.plot(ax=ax, color="#31a354", edgecolor="none", alpha=0.8, zorder=2)
city.boundary.plot(ax=ax, color="#252525", linewidth=1.2, linestyle="--", zorder=3)

for _, row in big.iterrows():
    p = row.geometry.representative_point()          # always inside the polygon, unlike the centroid
    ax.text(p.x, p.y, row["name"], fontsize=8, ha="center", va="center", color="#00441b",
            path_effects=HALO, zorder=5)

ax.text(503500, 4478600, "West Lafayette", fontsize=11, style="italic", color="#252525", path_effects=HALO, zorder=5)
ax.text(512500, 4470500, "Lafayette", fontsize=11, style="italic", color="#252525", path_effects=HALO, zorder=5)
zoom(ax)
ax.set_axis_off()
plt.show()

`representative_point()` returns a point guaranteed to lie inside the polygon; a centroid can fall outside an
L-shaped park. Place names are set by hand in italics, a common convention for areas and water. If labels
collide, label fewer features, shorten names, or move a label by adding to its coordinates.

## Step 4: A basemap

**contextily** downloads map tiles and draws them under your layers. Two rules:

- Pass the layer's CRS (`crs=tracts.crs`) so the tiles are reprojected to match, or project your data to Web
  Mercator (EPSG:3857).
- Choose a **quiet** basemap for thematic maps. Esri's World Gray Canvas loads in every notebook service (the OpenStreetMap and
  CARTO tile servers often refuse requests from notebook services).

Tiles carry their own attribution text, which contextily adds in the corner. Keep it: it is a license condition.

In [ ]:
import contextily as ctx

def add_basemap(ax):
    try:
        ctx.add_basemap(ax, crs=tracts.crs, source=ctx.providers.Esri.WorldGrayCanvas, zoom=13)
    except Exception as err:                              # no internet, or the tile server is down
        print("Basemap skipped:", type(err).__name__)

## Step 5: A legend for every layer

`legend=True` on the choropleth makes its own legend. Parks and the city limits need entries too. Build the
handles yourself and combine them with the choropleth classes, which mapclassify can compute directly.

In [ ]:
import mapclassify
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

classes = mapclassify.NaturalBreaks(tracts["pct_no_vehicle"].dropna(), k=5)
bins = [0] + list(classes.bins)
cmap = plt.get_cmap("YlOrRd")
colors = [cmap(i / 4) for i in range(5)]

handles = [Patch(fc=colors[i], ec="white", label=f"{bins[i]:.0f} to {bins[i + 1]:.0f}%") for i in range(5)]
handles += [Patch(fc="lightgray", hatch="///", ec="white", label="No households"),
            Patch(fc="#31a354", alpha=0.8, label="Park"),
            Line2D([], [], color="#252525", linestyle="--", label="City limits")]

## Step 6: Put it together

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
tracts.plot(ax=ax, column="pct_no_vehicle", scheme="UserDefined",
            classification_kwds={"bins": classes.bins}, cmap="YlOrRd", alpha=0.85,
            edgecolor="white", linewidth=0.4, zorder=2,
            missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white"})
parks.plot(ax=ax, color="#31a354", edgecolor="none", alpha=0.8, zorder=3)
city.boundary.plot(ax=ax, color="#252525", linewidth=1.2, linestyle="--", zorder=4)
for _, row in big.iterrows():
    p = row.geometry.representative_point()
    ax.text(p.x, p.y, row["name"], fontsize=8, ha="center", va="center", color="#00441b",
            path_effects=HALO, zorder=6)
zoom(ax)
add_basemap(ax)
ax.legend(handles=handles, title="Households without\na vehicle", loc="lower left", fontsize=8,
          title_fontsize=9, framealpha=0.9)
ax.set_title("Car-free households cluster near downtown Lafayette and campus", loc="left",
             fontsize=13, fontweight="bold")
ax.set_axis_off()
plt.show()

`scheme="UserDefined"` with the same `bins` guarantees the legend we built matches the map exactly, and
`alpha=0.85` lets a little of the basemap's street pattern show through the tracts.

## Try it yourself

Change the map so that **parks** are the subject and tracts are context:

1. Draw the tracts with a single light fill (no choropleth) and white outlines.
2. Draw parks in a strong green with a dark green outline.
3. Label the **eight** largest parks in view.
4. Update the title and legend.

<details>
<summary><b>Show a solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 9))
tracts.plot(ax=ax, color="#f0f0f0", edgecolor="white", linewidth=0.6, zorder=1)
parks.plot(ax=ax, color="#31a354", edgecolor="#00441b", linewidth=0.5, zorder=2)
city.boundary.plot(ax=ax, color="#737373", linewidth=1, linestyle="--", zorder=3)
for _, row in view.sort_values("ha", ascending=False).head(8).iterrows():
    p = row.geometry.representative_point()
    ax.text(p.x, p.y, row["name"], fontsize=8, ha="center", color="#00441b", path_effects=HALO, zorder=5)
zoom(ax)
ax.legend(handles=[Patch(fc="#31a354", ec="#00441b", label="Park"),
                   Line2D([], [], color="#737373", linestyle="--", label="City limits")], loc="lower left")
ax.set_title("Parks in Lafayette and West Lafayette", loc="left", fontsize=13, fontweight="bold")
ax.set_axis_off()
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- Draw layers in a deliberate order (`zorder`); use transparency or outlines for layers on top
- Zoom to the area that carries the story
- Label sparingly, with halos (`path_effects`) and `representative_point()`
- Add a quiet basemap with `contextily` in the data's CRS, and keep its attribution
- Build legend handles yourself when a map has several layers

**Next:** [02b Insets, Small Multiples and Export](02b_Insets_Small_Multiples_and_Export.ipynb)